# Player isolation across all camera angles

This notebook runs the reusable player-isolation pipeline for videos named with the pattern `<player>-<angle>`, such as `WP12-0.MP4`, `WP12-45.MP4`, and `WP12-135.MP4`.

For every video, the pipeline:

1. detects the competition mat,
2. selects the athlete nearest the mat center,
3. tracks that athlete through the video with ByteTrack,
4. isolates the athlete with a YOLO segmentation mask, and
5. restructures the processed dataset by player and camera angle.

```text
data/processed/players/
├── manifest.json
└── WP12/
    ├── manifest.json
    └── angles/
        ├── 0/
        │   ├── isolated.mp4
        │   ├── crop.mp4
        │   ├── tracking.csv
        │   └── metadata.json
        ├── 45/
        └── 135/
```


In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_USE_CUDA_DSA'] = "1" 

import torch
torch.cuda.is_available()

True

In [2]:
from dataclasses import asdict
from pathlib import Path
import sys

import cv2
import pandas as pd
from IPython.display import Video, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from poomsae_scoring.device import describe_inference_device
from poomsae_scoring.preprocessing.isolation import (
    IsolationConfig,
    PlayerIsolationPipeline,
    discover_player_videos,
    group_player_videos,
)


## Configuration

Set `DEVICE` to `"cuda:0"` for the first GPU, `"cuda:1"` for the second GPU, `"cpu"` to force CPU processing, or `"auto"` to use CUDA when available.

`MAX_DURATION_SECONDS` is useful for a short validation run. Set it to `None` before processing the complete dataset.


In [3]:
RAW_ROOT = PROJECT_ROOT / "data/raw"
OUTPUT_ROOT = PROJECT_ROOT / "data/processed/players"
MODEL_PATH = PROJECT_ROOT / "notebooks/yolo26n-seg.pt"

DEVICE = "auto"
CONFIDENCE = 0.35
RECOVERY_CONFIDENCE = 0.15
OUTPUT_SIZE = 640
RECOVERY_IMAGE_SIZE = 960
MAX_TRACKING_GAP = 8
MAX_DURATION_SECONDS = None

config = IsolationConfig(
    model_path=MODEL_PATH,
    device=DEVICE,
    confidence=CONFIDENCE,
    recovery_confidence=RECOVERY_CONFIDENCE,
    output_size=OUTPUT_SIZE,
    recovery_image_size=RECOVERY_IMAGE_SIZE,
    maximum_tracking_gap=MAX_TRACKING_GAP,
    maximum_duration_seconds=MAX_DURATION_SECONDS,
)

assert RAW_ROOT.is_dir(), f"Raw video directory does not exist: {RAW_ROOT}"
assert MODEL_PATH.is_file(), f"Segmentation model does not exist: {MODEL_PATH}"
print(f"Raw videos: {RAW_ROOT}")
print(f"Processed output: {OUTPUT_ROOT}")


Raw videos: /home/nazi/Desktop/poomsae-scoring-system/data/raw
Processed output: /home/nazi/Desktop/poomsae-scoring-system/data/processed/players


## Discover players and angles

Video identities come from filenames, not from appearance matching between cameras. All files with the same player prefix are grouped together, and the numeric suffix is used as the camera angle.


In [4]:
videos = discover_player_videos(RAW_ROOT)
players = group_player_videos(videos)

inventory = pd.DataFrame(
    [
        {
            "player_id": video.player_id,
            "angle": video.angle,
            "video_path": str(video.path),
        }
        for video in videos
    ]
)

expected_angles = {"0", "45", "135"}
missing_angles = {
    player_id: sorted(expected_angles - {video.angle for video in player_videos})
    for player_id, player_videos in players.items()
    if expected_angles - {video.angle for video in player_videos}
}

print(f"Found {len(videos)} videos for {len(players)} players.")
print(f"Missing angle sets: {missing_angles or 'none'}")
display(inventory.head(12))


Found 162 videos for 54 players.
Missing angle sets: none


,player_id,angle,video_path
0,WP1,0,/home/nazi/Desktop/poomsae-scoring-system/data...
1,WP1,45,/home/nazi/Desktop/poomsae-scoring-system/data...
2,WP1,135,/home/nazi/Desktop/poomsae-scoring-system/data...
3,WP2,0,/home/nazi/Desktop/poomsae-scoring-system/data...
4,WP2,45,/home/nazi/Desktop/poomsae-scoring-system/data...
5,WP2,135,/home/nazi/Desktop/poomsae-scoring-system/data...
6,WP3,0,/home/nazi/Desktop/poomsae-scoring-system/data...
7,WP3,45,/home/nazi/Desktop/poomsae-scoring-system/data...
8,WP3,135,/home/nazi/Desktop/poomsae-scoring-system/data...
9,WP4,0,/home/nazi/Desktop/poomsae-scoring-system/data...


## Preview one source video

This confirms the selected player and angle before running inference.


In [5]:
PREVIEW_PLAYER = next(iter(players))
PREVIEW_ANGLE = players[PREVIEW_PLAYER][0].angle
preview_video = next(
    video
    for video in players[PREVIEW_PLAYER]
    if video.angle == PREVIEW_ANGLE
)

capture = cv2.VideoCapture(str(preview_video.path))
preview_metadata = {
    "player_id": preview_video.player_id,
    "angle": preview_video.angle,
    "fps": capture.get(cv2.CAP_PROP_FPS),
    "frames": int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
    "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
}
capture.release()

display(pd.DataFrame([preview_metadata]))
display(Video(str(preview_video.path), width=800, embed=False))


,player_id,angle,fps,frames,width,height
0,WP1,0,50.0,2609,1920,1080


## Optional short smoke test

The smoke test processes one video into a separate `_smoke` directory. Enable it before a full batch when testing a new machine, model, or camera setup.


In [6]:
RUN_SMOKE_TEST = False
SMOKE_DURATION_SECONDS = 0.5

if RUN_SMOKE_TEST:
    smoke_config = IsolationConfig(
        model_path=MODEL_PATH,
        device=DEVICE,
        confidence=CONFIDENCE,
        recovery_confidence=RECOVERY_CONFIDENCE,
        output_size=320,
        recovery_image_size=480,
        maximum_tracking_gap=MAX_TRACKING_GAP,
        maximum_duration_seconds=SMOKE_DURATION_SECONDS,
    )
    smoke_pipeline = PlayerIsolationPipeline(smoke_config)
    print(describe_inference_device(smoke_pipeline.device))
    smoke_result = smoke_pipeline.process_video(
        preview_video,
        OUTPUT_ROOT / "_smoke",
    )
    display(pd.DataFrame([asdict(smoke_result)]))
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True to run it.")


Smoke test skipped. Set RUN_SMOKE_TEST = True to run it.


## Process three presentations and every angle

Each run processes at most three incomplete player presentations. A presentation is skipped when all expected output files already exist for every available angle. For a partially completed presentation, only its missing angles are processed.


In [ ]:
RUN_BATCH = True
PRESENTATIONS_PER_BATCH = 1
REQUIRED_OUTPUT_FILES = ("metadata.json", "tracking.csv", "isolated.mp4", "crop.mp4")


def angle_is_complete(video):
    output_directory = OUTPUT_ROOT / video.player_id / "angles" / video.angle
    return all((output_directory / filename).is_file() for filename in REQUIRED_OUTPUT_FILES)


pending_presentations = [
    (player_id, player_videos)
    for player_id, player_videos in players.items()
    if not all(angle_is_complete(video) for video in player_videos)
]
selected_presentations = pending_presentations[:PRESENTATIONS_PER_BATCH]

if RUN_BATCH and selected_presentations:
    pipeline = PlayerIsolationPipeline(config)
    print(describe_inference_device(pipeline.device))
    print(
        "Processing presentations: "
        + ", ".join(player_id for player_id, _ in selected_presentations)
    )
    results = []
    skipped_angles = []
    for player_id, player_videos in selected_presentations:
        for video in player_videos:
            if angle_is_complete(video):
                skipped_angles.append(f"{player_id}-{video.angle}")
                continue
            results.append(pipeline.process_video(video, OUTPUT_ROOT))

    if skipped_angles:
        print("Skipped existing angles: " + ", ".join(skipped_angles))

    results_table = pd.DataFrame(
        [
            {
                "player_id": result.player_id,
                "angle": result.angle,
                "frames_processed": result.frames_processed,
                "frames_isolated": result.frames_isolated,
                "frames_recovered": result.frames_recovered,
                "isolation_rate": (
                    result.frames_isolated / result.frames_processed
                    if result.frames_processed
                    else 0.0
                ),
                "output_directory": result.output_directory,
            }
            for result in results
        ]
    )
    display(results_table)
elif RUN_BATCH:
    print("All discovered presentations are already complete.")
else:
    print("Batch processing skipped. Set RUN_BATCH = True to run it.")


Inference device: cuda:0 (NVIDIA GeForce RTX 3060 Laptop GPU)
Processing presentations: WP4, WP5, WP6


Isolating WP6-135.MP4: 100%|██████████| 2385/2385 [01:44<00:00, 22.81frame/s]


Skipped existing angles: WP4-0, WP4-45


,player_id,angle,frames_processed,frames_isolated,frames_recovered,isolation_rate,output_directory
0,WP4,135,2336,0,0,0.000000,/home/nazi/Desktop/poomsae-scoring-system/data...
1,WP5,0,2417,2416,19,0.999586,/home/nazi/Desktop/poomsae-scoring-system/data...
2,WP5,45,2418,2418,10,1.000000,/home/nazi/Desktop/poomsae-scoring-system/data...
3,WP5,135,2413,1843,5,0.763780,/home/nazi/Desktop/poomsae-scoring-system/data...
4,WP6,0,2389,2079,13,0.870239,/home/nazi/Desktop/poomsae-scoring-system/data...
5,WP6,45,2366,2366,2,1.000000,/home/nazi/Desktop/poomsae-scoring-system/data...
6,WP6,135,2385,2384,5,0.999581,/home/nazi/Desktop/poomsae-scoring-system/data...


## Inspect processed results

After the batch finishes, select a player and angle to inspect tracking quality and preview the isolated and normalized videos.


In [10]:
RESULT_PLAYER = PREVIEW_PLAYER
RESULT_ANGLE = PREVIEW_ANGLE
result_directory = OUTPUT_ROOT / RESULT_PLAYER / "angles" / RESULT_ANGLE
tracking_path = result_directory / "tracking.csv"
isolated_path = result_directory / "isolated.mp4"
crop_path = result_directory / "crop.mp4"

if tracking_path.is_file():
    tracking = pd.read_csv(tracking_path)
    display(tracking.head())
    display(tracking["status"].value_counts(dropna=False).rename("frames"))
    display(Video(str(isolated_path), width=800, embed=False))
    display(Video(str(crop_path), width=500, embed=False))
else:
    print(f"No processed result exists yet: {result_directory}")


,frame,track_id,confidence,x1,y1,x2,y2,status
0,0,1.0,0.882958,929.756836,251.019150,1098.933838,729.908997,tracked
1,1,1.0,0.877657,929.870239,250.508652,1098.949341,730.238892,tracked
2,2,1.0,0.875566,929.295959,250.547913,1098.685913,730.083313,tracked
3,3,1.0,0.861184,929.478027,250.431015,1098.766113,729.929443,tracked
4,4,1.0,0.849223,929.562378,249.889481,1098.905396,730.204651,tracked


status
tracked      2591
recovered      16
Name: frames, dtype: int64